In [1]:
!pip install -U albumentations==1.4.10 albucore==0.0.12 opencv-python==4.10.0.84 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.9/161.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 28.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 94.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.0/236.0 kB 18.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
category-encoders 2.7.0 requires scikit-learn<1.6.0,>=1.0.0, but you have scikit-learn 1.7.1 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
sklearn-compat 0.1.3 requires scikit-learn<1.7,>=1.2, but you have scikit-learn 1.7.1 which is incompatible.


In [18]:
import os, math, random, gc, warnings, json, time
from pathlib import Path

import numpy as np
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from albumentations import (
    Compose, HorizontalFlip, RandomBrightnessContrast,
    GaussianBlur, ShiftScaleRotate, Resize, Normalize
)
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

BASE_OUT   = Path("/kaggle/working/outputs")
EXP_NAME   = "C_dilate3"
EXP_DIR    = BASE_OUT / EXP_NAME
SEEDS_DIR  = BASE_OUT / "seed_overlays" / "gradcam"        # optional overlays
PSEUDO_SRC = BASE_OUT / "pseudo_masks" / "gradcam"         # baseline Grad-CAM (binary)
PSEUDO_DIR = BASE_OUT / "pseudo_masks" / "gradcam_dilate3" # dilated(3) version for Exp C
RESULTS_CSV= EXP_DIR / "voc_val_robustness.csv"
CKPT_PATH  = EXP_DIR / "deeplab_binary_best.pth"
for d in [EXP_DIR, SEEDS_DIR, PSEUDO_SRC, PSEUDO_DIR]: d.mkdir(parents=True, exist_ok=True)

Device: cuda


In [20]:
VOC_ROOT   = Path("/kaggle/input/pascal-voc-2012-dataset/VOC2012_train_val/VOC2012_train_val")
IMG_ROOT   = VOC_ROOT / "JPEGImages"
GT_ROOT    = VOC_ROOT / "SegmentationClass"
SPLIT_ROOT = VOC_ROOT / "ImageSets" / "Segmentation"

def read_ids(txt: Path):
    return [x.strip() for x in open(txt) if x.strip()]

assert VOC_ROOT.exists(),   f"Missing {VOC_ROOT}"
assert IMG_ROOT.exists(),   "JPEGImages not found"
assert GT_ROOT.exists(),    "SegmentationClass not found"
assert SPLIT_ROOT.exists(), "ImageSets/Segmentation not found"

train_ids = read_ids(SPLIT_ROOT/"train.txt")
val_ids   = read_ids(SPLIT_ROOT/"val.txt")
print(f"IDs → train={len(train_ids)}  val={len(val_ids)} (expect ~1464 / 1449)")
print("Sample ids:", train_ids[:5])

IDs → train=1464  val=1449 (expect ~1464 / 1449)
Sample ids: ['2007_000032', '2007_000039', '2007_000063', '2007_000068', '2007_000121']


In [22]:
import torchvision
from torchvision.models import resnet50, ResNet50_Weights

class CAMHelper:
    """Single-use Grad-CAM on resnet50 final conv."""
    def __init__(self):
        try:
            m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        except Exception:
            m = resnet50(weights=None)
        self.model = m.to(DEVICE).eval()
        self.feats, self.grads = [], []

        def f_hook(_, __, out): self.feats.append(out.detach())
        def b_hook(_, grad_in, grad_out): self.grads.append(grad_out[0].detach())

        self.handles = [
            self.model.layer4[-1].conv3.register_forward_hook(f_hook),
            self.model.layer4[-1].conv3.register_full_backward_hook(b_hook),
        ]
        self.pre = torchvision.transforms.Compose([
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize(mean=[0.485,0.456,0.406],
                                             std=[0.229,0.224,0.225])
        ])

    @torch.inference_mode(False)
    def __call__(self, pil_img: Image.Image) -> np.ndarray:
        self.feats.clear(); self.grads.clear()
        x = self.pre(pil_img).unsqueeze(0).to(DEVICE)
        x.requires_grad_(True)
        logits = self.model(x)
        c = logits.argmax(dim=1)
        logits[0, c].backward()
        A = self.feats[-1][0]      # [C,H,W]
        G = self.grads[-1][0]      # [C,H,W]
        w = G.mean(dim=(1,2))      # [C]
        cam = (w[:,None,None] * A).sum(0)
        cam = torch.relu(cam)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-6)
        return cam.detach().cpu().numpy()

    def close(self):
        for h in self.handles: h.remove()

def build_gradcam_masks(ids, th=0.30):
    cam = CAMHelper()
    wrote, non_empty = 0, 0
    for img_id in tqdm(ids, desc="Grad-CAM → pseudo", unit="img"):
        ip = (IMG_ROOT / f"{img_id}.jpg")
        if not ip.exists(): ip = IMG_ROOT / f"{img_id}.jpeg"
        img = Image.open(ip).convert("RGB")
        W,H = img.size

        # get CAM and upsample to image size
        cmap = cam(img)
        cam_up = cv2.resize(cmap, (W,H), interpolation=cv2.INTER_LINEAR)

        # threshold to binary mask
        mask = (cam_up >= th).astype(np.uint8) * 255
        if mask.any(): non_empty += 1

        Image.fromarray(mask).save(PSEUDO_SRC / f"{img_id}.png")
        wrote += 1

    cam.close()
    print(f"Wrote {wrote} masks to {PSEUDO_SRC}  | non-empty: {non_empty}/{len(ids)}")


if len(list(PSEUDO_SRC.glob("*.png"))) < len(train_ids):
    build_gradcam_masks(train_ids, th=0.30)
else:
    print("Grad-CAM masks already present. Skipping rebuild.")

Grad-CAM → pseudo:   0%|          | 0/1464 [00:00<?, ?img/s]

Wrote 1464 masks to /kaggle/working/outputs/pseudo_masks/gradcam  | non-empty: 1464/1464


In [23]:
paths = sorted(PSEUDO_SRC.glob("*.png"))
print("num Grad-CAM masks:", len(paths))
nonempty = sum((np.array(Image.open(p))>0).any() for p in paths[:200])
print(f"non-empty in first 200: {nonempty}")
assert len(paths) >= len(train_ids)*0.95, "Fewer seeds than expected."

num Grad-CAM masks: 1464
non-empty in first 200: 200


In [24]:
def build_dilate3(src_dir: Path, dst_dir: Path, ids, kernel_radius_px=3):
    ksize = int(kernel_radius_px)*2 + 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksize, ksize))
    wrote, missing = 0, 0

    for img_id in tqdm(ids, desc="Dilate x3", unit="mask"):
        src = src_dir / f"{img_id}.png"
        if not src.exists():
            missing += 1
            continue
        arr = np.array(Image.open(src))
        if arr.ndim == 3: arr = arr[...,0]
        fg = (arr > 0).astype(np.uint8)
        dil = cv2.dilate(fg, kernel, iterations=1) * 255
        Image.fromarray(dil.astype(np.uint8)).save(dst_dir / f"{img_id}.png")
        wrote += 1

    print(f"dilate3 → wrote {wrote} masks to {dst_dir}")
    if missing:
        print(f"WARNING: {missing} masks were missing in {src_dir}")

build_dilate3(PSEUDO_SRC, PSEUDO_DIR, train_ids, kernel_radius_px=3)

Dilate x3:   0%|          | 0/1464 [00:00<?, ?mask/s]

dilate3 → wrote 1464 masks to /kaggle/working/outputs/pseudo_masks/gradcam_dilate3


In [27]:
IGNORE_IDX = 255
IMG_SIZE   = 256
BATCH_TRAIN= 8
BATCH_VAL  = 8

class VOCPseudoBinary(Dataset):
    """
    x: FloatTensor [3,H,W]
    g: LongTensor  [H,W] in {0,1}      (pseudo)
    q: LongTensor  [H,W] in {0,1,255}  (GT binary-for-eval; 255 ignore)
    id: str
    """
    def __init__(self, ids, img_root, pseudo_root, gt_root, train=True, size=256):
        self.ids         = ids
        self.img_root    = Path(img_root)
        self.pseudo_root = Path(pseudo_root)
        self.gt_root     = Path(gt_root)
        self.train       = bool(train)
        self.size        = int(size)

        aug_train = Compose([
            HorizontalFlip(p=0.5),
            RandomBrightnessContrast(p=0.2),
            GaussianBlur(blur_limit=(3,5), p=0.15),
            ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15,
                             border_mode=cv2.BORDER_CONSTANT, p=0.5),
            Resize(self.size, self.size),
            Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2()
        ])
        aug_val = Compose([
            Resize(self.size, self.size),
            Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2()
        ])
        self.tr = aug_train if self.train else aug_val

    def __len__(self): return len(self.ids)

    def __getitem__(self, i):
        img_id = self.ids[i]

        # image
        ip = self.img_root / f"{img_id}.jpg"
        if not ip.exists(): ip = self.img_root / f"{img_id}.jpeg"
        img = np.array(Image.open(ip).convert("RGB"))
        H, W = img.shape[:2]

        # pseudo (binary 0/255)
        pp = self.pseudo_root / f"{img_id}.png"
        if pp.exists(): pm = np.array(Image.open(pp))
        else:           pm = np.zeros((H,W), np.uint8)
        if pm.ndim==3: pm = pm[...,0]
        if pm.shape!=(H,W): pm = cv2.resize(pm, (W,H), interpolation=cv2.INTER_NEAREST)

        # GT (21-class; we eval as fg = any class except 0 & 255)
        gp = self.gt_root / f"{img_id}.png"
        if gp.exists(): gt = np.array(Image.open(gp))
        else:           gt = np.full((H,W), IGNORE_IDX, np.uint8)
        if gt.shape!=(H,W): gt = cv2.resize(gt, (W,H), interpolation=cv2.INTER_NEAREST)
        gbin = np.where(gt==IGNORE_IDX, IGNORE_IDX, (gt!=0).astype(np.uint8))  # 0/1/255

        out = self.tr(image=img, mask=pm, masks=[gbin])
        x   = out["image"].float()
        pm2 = out["mask"]       # tensor [H,W] float or uint8
        gt2 = out["masks"][0]

        pm2 = torch.as_tensor(pm2).squeeze()
        if pm2.dtype.is_floating_point:
            g = (pm2 > 0.5).long()
        else:
            g = (pm2 > 0).long()

        q = torch.as_tensor(gt2).squeeze().long()
        return x, g, q, img_id

def make_loaders(num_workers=2, pin=True):
    train_ds = VOCPseudoBinary(train_ids, IMG_ROOT, PSEUDO_DIR, GT_ROOT, train=True,  size=IMG_SIZE)
    val_ds   = VOCPseudoBinary(val_ids,   IMG_ROOT, PSEUDO_DIR, GT_ROOT, train=False, size=IMG_SIZE)
    train_dl = DataLoader(train_ds, batch_size=BATCH_TRAIN, shuffle=True,  num_workers=num_workers, pin_memory=pin, persistent_workers=False)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_VAL,   shuffle=False, num_workers=num_workers, pin_memory=pin, persistent_workers=False)
    return train_dl, val_dl

train_dl, val_dl = make_loaders()
bx, bg, bq, bids = next(iter(train_dl))
print("Batch shapes:", tuple(bx.shape), tuple(bg.shape), tuple(bq.shape))

probe_dl = DataLoader(train_dl.dataset, batch_size=32, shuffle=False, num_workers=0)
fg_pix=0; tot_pix=0
for _,g,_,_ in probe_dl:
    fg_pix  += (g>0).sum().item()
    tot_pix += g.numel()
    if tot_pix > 2*IMG_SIZE*IMG_SIZE*200: break
ratio = fg_pix/max(1,tot_pix)
print(f"[probe] pseudo>0 pixels: {fg_pix} ({ratio:.2%} of probed)")

Batch shapes: (8, 3, 256, 256) (8, 256, 256) (8, 256, 256)
[probe] pseudo>0 pixels: 7770860 (28.50% of probed)


In [29]:
train_dl, val_dl = make_loaders(num_workers=0, pin=False)
print(len(train_dl), len(val_dl))

183 182


In [30]:
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

def build_deeplab_binary(num_classes=2):
    try:
        m = deeplabv3_resnet50(weights_backbone=DeepLabV3_ResNet50_Weights.IMAGENET1K_V1)
    except Exception:
        m = deeplabv3_resnet50(weights_backbone=None)
    m.classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
    return m.to(DEVICE)

def dice_loss_fg(logits, target01, eps=1e-6):
    probs = torch.sigmoid(logits[:,1])  # foreground channel
    tgt   = target01.float()
    inter = (probs*tgt).sum(dim=(1,2))
    denom = (probs.sum(dim=(1,2)) + tgt.sum(dim=(1,2)) + eps)
    dice  = (2*inter + eps) / denom
    return (1 - dice).mean()

def ce_loss_2c(logits, target_q):
    return F.cross_entropy(logits, target_q.clamp(0,1), ignore_index=IGNORE_IDX)

@torch.no_grad()
def evaluate(model, dl):
    model.eval()
    inter_bg=inter_fg=union_bg=union_fg=0
    for x,g,q,_ in tqdm(dl, desc="val", leave=False):
        x=x.to(DEVICE); q=q.to(DEVICE)
        pred = model(x)["out"].argmax(1)   # [B,H,W] in {0,1}
        mask = (q!=IGNORE_IDX)
        pr = pred[mask]; gt = q[mask]
        for cls in [0,1]:
            pr_c=(pr==cls); gt_c=(gt==cls)
            inter = (pr_c & gt_c).sum().item()
            union = (pr_c | gt_c).sum().item()
            if cls==0: inter_bg+=inter; union_bg+=union
            else:      inter_fg+=inter; union_fg+=union
    iou_bg = inter_bg/max(1,union_bg)
    iou_fg = inter_fg/max(1,union_fg)
    miou   = 0.5*(iou_bg+iou_fg)
    return dict(iou_bg=iou_bg, iou_fg=iou_fg, miou=miou)

EPOCHS   = 15
PATIENCE = 5
LR       = 1e-3
WD       = 1e-4

model = build_deeplab_binary(num_classes=2)
opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

best = 0.0; wait = 0
try:
    for ep in range(1, EPOCHS+1):
        # IMPORTANT: keep loaders single-process every epoch
        tr_dl = DataLoader(train_dl.dataset, batch_size=BATCH_TRAIN, shuffle=True,
                           num_workers=0, pin_memory=False, persistent_workers=False)
        vl_dl = DataLoader(val_dl.dataset,   batch_size=BATCH_VAL,   shuffle=False,
                           num_workers=0, pin_memory=False, persistent_workers=False)

        model.train()
        total=0.0
        pbar = tqdm(tr_dl, desc=f"train ep{ep:02d}", leave=False)
        for x,g,q,_ in pbar:
            x=x.to(DEVICE); g=g.to(DEVICE); q=q.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            out  = model(x)["out"]
            loss = 0.5*dice_loss_fg(out, g) + 0.5*ce_loss_2c(out, q)
            loss.backward()
            opt.step()
            total += loss.item()*x.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        tr_loss = total/len(tr_dl.dataset)

        mets = evaluate(model, vl_dl)
        print(f"Epoch {ep:02d} | loss={tr_loss:.4f} | mIoU={mets['miou']:.3f} "
              f"(bg={mets['iou_bg']:.3f}, fg={mets['iou_fg']:.3f})")

        # checkpoint every epoch + keep best
        torch.save(model.state_dict(), CKPT_PATH.with_name(f"ep{ep:02d}.pth"))
        if mets["miou"] > best + 1e-4:
            best = mets["miou"]; wait = 0
            torch.save(model.state_dict(), CKPT_PATH)
            print("  New best; checkpoint saved:", CKPT_PATH.name)
        else:
            wait += 1
            if wait >= PATIENCE:
                print("  Early stop "); break

       
        del tr_dl, vl_dl; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

    print("Best mIoU:", round(best,3))

except KeyboardInterrupt:
    print("Interrupted — saving snapshot …")
    torch.save(model.state_dict(), CKPT_PATH.with_name("interrupt_snapshot.pth"))

train ep01:   0%|          | 0/183 [00:00<?, ?it/s]

val:   0%|          | 0/182 [00:00<?, ?it/s]

Epoch 01 | loss=0.5070 | mIoU=0.541 (bg=0.743, fg=0.339)
  ✓ New best; checkpoint saved: deeplab_binary_best.pth


train ep02:   0%|          | 0/183 [00:00<?, ?it/s]

val:   0%|          | 0/182 [00:00<?, ?it/s]

...

val:   0%|          | 0/182 [00:00<?, ?it/s]

Epoch 14 | loss=0.4200 | mIoU=0.598 (bg=0.779, fg=0.417)


train ep15:   0%|          | 0/183 [00:00<?, ?it/s]

val:   0%|          | 0/182 [00:00<?, ?it/s]

Epoch 15 | loss=0.4184 | mIoU=0.618 (bg=0.778, fg=0.458)
  ✓ New best; checkpoint saved: deeplab_binary_best.pth
Best mIoU: 0.618


In [31]:
import pandas as pd

def rotate_keep_size(img, deg=10):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), deg, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT101)

def perturbations(img):
    imgf = img.astype(np.float32)
    out = {
        "clean":      img,
        "blur":       cv2.GaussianBlur(img, (5,5), 1.0),
        "brightness": np.clip(imgf * 1.25, 0, 255).astype(np.uint8),
        "gauss":      np.clip(imgf + np.random.normal(0, 10, img.shape), 0, 255).astype(np.uint8),
        "hflip":      img[:, ::-1],
        "rotation":   rotate_keep_size(img, deg=10),
    }
    return out

@torch.no_grad()
def eval_under_perturbations(val_ids, max_items=None):
    rows=[]
    ids = val_ids if max_items is None else val_ids[:max_items]
    for pid, img_id in enumerate(tqdm(ids, desc="robustness", unit="img")):
        ip = (IMG_ROOT / f"{img_id}.jpg")
        if not ip.exists(): ip = IMG_ROOT / f"{img_id}.jpeg"
        img = np.array(Image.open(ip).convert("RGB"))
        H,W = img.shape[:2]

        # GT
        gp = GT_ROOT / f"{img_id}.png"
        gt = np.array(Image.open(gp)) if gp.exists() else np.full((H,W), IGNORE_IDX, np.uint8)

        for k,im in perturbations(img).items():
       
            tr = Compose([
                Resize(IMG_SIZE, IMG_SIZE),
                Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
                ToTensorV2()
            ])
            out = tr(image=im)
            x = out["image"].unsqueeze(0).to(DEVICE)
            pred = model(x)["out"].argmax(1)[0].detach().cpu().numpy()

           
            pred = cv2.resize(pred.astype(np.uint8), (W,H), interpolation=cv2.INTER_NEAREST)

           
            gbin = np.where(gt==IGNORE_IDX, IGNORE_IDX, (gt!=0).astype(np.uint8))
            m = (gbin!=IGNORE_IDX)
            pr = pred[m]; gt_b = gbin[m]

          
            def iou_of(cls):
                pr_c = (pr==cls); gt_c = (gt_b==cls)
                inter = (pr_c & gt_c).sum()
                union = (pr_c | gt_c).sum()
                return inter / max(1, union)
            iou_bg = iou_of(0); iou_fg = iou_of(1)
            rows.append([k, iou_bg, iou_fg])
    df = pd.DataFrame(rows, columns=["perturb","IoU_bg","IoU_fg"])
    df = df.groupby("perturb").mean().reset_index()
    df["mIoU"] = 0.5*(df["IoU_bg"]+df["IoU_fg"])
    df = df[["perturb","IoU_bg","IoU_fg","mIoU"]]
    df = df.sort_values("perturb").reset_index(drop=True)
    df.round(6).to_csv(RESULTS_CSV, index=False)
    print("Saved:", RESULTS_CSV)
    return df.round(6)

df_rob = eval_under_perturbations(val_ids, max_items=None)  
df_rob

robustness:   0%|          | 0/1449 [00:00<?, ?img/s]

Saved: /kaggle/working/outputs/C_dilate3/voc_val_robustness.csv


,perturb,IoU_bg,IoU_fg,mIoU
0,blur,0.757293,0.469410,0.613352
1,brightness,0.762195,0.462364,0.612280
2,clean,0.762563,0.462444,0.612503
3,gauss,0.762434,0.461958,0.612196
4,hflip,0.702389,0.318175,0.510282
5,rotation,0.740060,0.421399,0.580730
